# Delta Hedging Simulation for European Options

This notebook presents the full delta hedging workflow used in the project. It follows the project statement directly: simulate stock price paths, sell an option and hedge with delta, rebalance periodically, and track the hedging error over time.

## Project Statement

Delta hedging attempts to neutralize price sensitivity.

**Tasks**

1. Simulate stock price paths.
2. Sell an option and hedge using the delta of the option.
3. Rebalance the hedge periodically.
4. Track hedging error over time.

In [ ]:
from pathlib import Path

from src.bs_model import bs_call_price, bs_delta
from src.experiments import (
    experiment_hedge_frequency,
    experiment_moneyness,
    experiment_volatility,
)
from src.gbm import GBMConfig, plot_sample_paths, simulate_gbm_paths, validate_gbm_paths
from src.hedging import delta_hedge_single_path, run_delta_hedge

Path('imgs').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)

## 1. Simulating Stock Price Paths

We model the stock under Geometric Brownian Motion and validate the simulated terminal distribution against the theoretical mean and variance.

In [1]:
cfg = GBMConfig(
    S0=100.0,
    r=0.05,
    sigma=0.2,
    T=1.0,
    N=252,
    M=10000,
    seed=42,
)

paths = simulate_gbm_paths(cfg.S0, cfg.r, cfg.sigma, cfg.T, cfg.N, cfg.M, cfg.seed)
gbm_summary = validate_gbm_paths(paths, cfg.S0, cfg.r, cfg.sigma, cfg.T)

for key, value in gbm_summary.items():
    print(f"{key}: {value:.6f}")

GBM Validation
empirical_mean: 105.315741
theoretical_mean: 105.127110
empirical_variance: 455.772891
theoretical_variance: 451.028808
mean_error: 0.188631
variance_error: 4.744083
relative_mean_error: 0.001794
relative_variance_error: 0.010518


### Simulated GBM Paths

![Simulated GBM Paths](imgs/sample_gbm_paths.png)

These are sample stock price paths generated under the Geometric Brownian Motion model. The spread widens over time because uncertainty accumulates as the horizon increases.

## 2. Selling the Option and Computing Delta

The option is priced using the Black-Scholes formula. Its delta gives the hedge ratio, which tells us how many shares are needed to offset small stock-price movements.

In [2]:
K = 100.0
initial_call_price = bs_call_price(cfg.S0, K, cfg.T, cfg.r, cfg.sigma)
initial_delta = bs_delta(cfg.S0, K, cfg.T, cfg.r, cfg.sigma)

print(f"initial_call_price: {initial_call_price:.6f}")
print(f"initial_delta: {initial_delta:.6f}")

Black-Scholes Checks
initial_call_price: 10.450584
initial_delta: 0.636831
min_price: 0.002399
max_price: 54.970140
min_delta: 0.000917
max_delta: 0.991281
price_increases_with_stock: True
delta_between_zero_and_one: True


The initial call price is about **10.4506** and the initial delta is about **0.6368**. This means that after selling one call option, we hedge by holding roughly 0.64 shares of stock.

## 3. Rebalancing the Hedge Periodically

Once the option is sold, the hedge is updated at discrete times. At every rebalancing step, we recompute the delta using the current stock price and the remaining time to maturity.

In [3]:
hedge_data = run_delta_hedge(paths, K, cfg.T, cfg.r, cfg.sigma)
single_path_data = delta_hedge_single_path(paths[0], K, cfg.T, cfg.r, cfg.sigma, cfg.T / cfg.N)

for key, value in hedge_data['summary'].items():
    print(f"{key}: {value:.6f}")

Delta Hedging Summary
mean_hedging_error: 0.001860
std_hedging_error: 0.436352
min_hedging_error: -2.996266
max_hedging_error: 1.969362
rmse_hedging_error: 0.436334
mean_final_pnl: 0.001860
probability_of_loss: 0.497300


### Single Path Diagnostics

![Single Path Diagnostics](imgs/single_path_diagnostics.png)

The top panel shows one simulated stock path. The middle panel shows how delta changes through time. The bottom panel shows the corresponding hedging portfolio value, which moves away from zero because the hedge is rebalanced only at discrete dates.

## 4. Tracking Hedging Error Over Time

The final hedging error is the difference between the terminal hedge value and the option payoff. A good hedge should have mean error close to zero, but discrete trading still leaves a nonzero spread of outcomes.

### Portfolio Value Through Time

![Portfolio Value Through Time](imgs/portfolio_value_paths.png)

Each line is one hedged portfolio path. The trajectories remain controlled, but they do not stay exactly at zero. This is the pathwise effect of discrete hedging error.

### Average Hedging Error Through Time

![Average Hedging Error Through Time](imgs/average_hedging_error.png)

The average error across all simulated paths stays close to zero. This supports the idea that the hedge is approximately unbiased under the model assumptions.

### Distribution of Hedging Error

![Distribution of Hedging Error](imgs/hedging_error_histogram.png)

The histogram is centered close to zero, but it still has visible spread. That spread is the residual error caused by discrete rebalancing.

## Sensitivity Analysis

We test how the hedge behaves when rebalancing frequency, volatility, and option moneyness are changed.

In [4]:
frequency_results = experiment_hedge_frequency(cfg.S0, K, cfg.T, cfg.r, cfg.sigma, cfg.M, seed=cfg.seed)
for label, stats in frequency_results.items():
    print(label)
    for key, value in stats.items():
        print(f"{key}: {value:.6f}")
    print()

Rebalancing Frequency

Daily
mean: 0.001860
std: 0.436352
rmse: 0.436334
mean_abs_error: 0.325384

Weekly
mean: -0.006297
std: 0.950706
rmse: 0.950679
mean_abs_error: 0.710923

Monthly
mean: -0.028095
std: 1.958420
rmse: 1.958524
mean_abs_error: 1.471481


### Rebalancing Frequency and Hedging Error

![Rebalancing Frequency and Hedging Error](imgs/frequency_vs_error.png)

Daily hedging has the lowest error standard deviation, while weekly and monthly hedging have larger error. This matches theory: more frequent rebalancing keeps the hedge closer to the required delta.

In [5]:
volatility_results = experiment_volatility(cfg.S0, K, cfg.T, cfg.r, cfg.N, cfg.M, seed=cfg.seed)
for label, stats in volatility_results.items():
    print(label)
    for key, value in stats.items():
        print(f"{key}: {value:.6f}")
    print()

Volatility Scenarios

Low Volatility
mean: 0.001713
std: 0.323270
rmse: 0.323258
mean_abs_error: 0.239780

High Volatility
mean: 0.003614
std: 0.765680
rmse: 0.765650
mean_abs_error: 0.573546


### Volatility and Hedging Error

![Volatility and Hedging Error](imgs/volatility_vs_error.png)

The high-volatility case shows much larger hedging error. When the stock moves more sharply between rebalancing times, the delta becomes outdated more quickly.

In [6]:
moneyness_results = experiment_moneyness(cfg.S0, cfg.T, cfg.r, cfg.sigma, cfg.N, cfg.M, seed=cfg.seed)
for label, stats in moneyness_results.items():
    print(label)
    for key, value in stats.items():
        print(f"{key}: {value:.6f}")
    print()

Moneyness Scenarios

ITM
mean: 0.000013
std: 0.340166
rmse: 0.340149
mean_abs_error: 0.240427

ATM
mean: 0.001860
std: 0.436352
rmse: 0.436334
mean_abs_error: 0.325384

OTM
mean: 0.003173
std: 0.459036
rmse: 0.459024
mean_abs_error: 0.340234


### Moneyness and Hedging Error

![Moneyness and Hedging Error](imgs/moneyness_vs_error.png)

Hedging performance changes with moneyness because delta behaves differently for ITM, ATM, and OTM options. In these results, OTM has a slightly larger error spread than ATM, while ITM is the most stable among the three.

## Conclusion

This notebook shows that delta hedging does reduce option risk, but it does not remove it perfectly when the hedge is rebalanced in discrete time. The experiments confirm the main theoretical trends: higher rebalancing frequency improves hedge quality, higher volatility increases error, and the option's moneyness affects hedge stability.